# For1Sight V2.8 – Stage 2: Pre-Race Feature Engineering

## Purpose
Prepare a modelling dataset that matches the information available to the future-race simulator.

**Prediction task:** one binary row per driver per Grand Prix; predict whether that driver wins the race.

**Scenario:** the user selects a future season and circuit, then supplies a hypothetical starting grid. The system uses the fixed 2026 driver roster and historical information from 2024–2026.

## Important scope decisions
- Historical source data: 2024–2026.
- Fixed simulator roster: latest 2026 event entry list (22 drivers, subject to roster validation).
- Future prediction season is independent of historical event records.
- No user-selected race-round input is required.
- Practice and qualifying measurements from the target event are excluded from this feature schema because the simulator does not collect them and a hypothetical grid may differ from actual qualifying order.
- The actual starting-grid position is retained as a feature because the user supplies this information.
- Historical driver, constructor and circuit features are retained as candidates, subject to the temporal-provenance checks in this notebook.

## Critical limitation
The V2.6 Parquet file contains precomputed `*_prior_*` fields, but this project ZIP does not include the ETL source that originally calculated them. This notebook checks their availability and missingness; it does **not** prove how those prior fields were calculated. Before treating them as fully verified, compare their construction with the original ETL logic or independently rebuild them from prior race results.

## 0. Environment and project paths

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\ayepd\Desktop\For1Sight Project")
SOURCE_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_6.parquet"
OUTPUT_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_8_prerace.parquet"
MANIFEST_PATH = PROJECT_ROOT / "for1sight_warehouse" / "prediction_training_v2_8_prerace_manifest.json"

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE_PATH}")

df = pd.read_parquet(SOURCE_PATH, engine="pyarrow")
print("Source:", SOURCE_PATH)
print("Shape:", df.shape)
print("Seasons:", sorted(pd.to_numeric(df["year"], errors="coerce").dropna().unique().tolist()))

Source: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_6.parquet
Shape: (1354, 49)
Seasons: [2024, 2025, 2026]


## 1. Recreate the agreed eligible event population

In [2]:
def normalize_event_name(value):
    value = str(value or "").lower().strip()
    value = re.sub(r"\\bgrand\\s*prix\\b", "", value)
    return re.sub(r"[^a-z0-9]", "", value)

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

required = [
    "year", "event_name", "event_norm", "driver_code", "driver_code_norm",
    "constructor", "starting_grid", "finishing_position", "is_winner",
    "event_prediction_ready", "model_training_eligible",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required source columns are missing: {missing}")

work = df.copy()
work["year"] = pd.to_numeric(work["year"], errors="coerce").astype("Int64")
work["target"] = pd.to_numeric(work["is_winner"], errors="coerce")
work["finishing_position_check"] = pd.to_numeric(work["finishing_position"], errors="coerce")
work["starting_grid"] = pd.to_numeric(work["starting_grid"], errors="coerce")
work["event_key"] = work["event_name"].map(normalize_event_name)

ready = to_bool(work["event_prediction_ready"])
train_eligible = to_bool(work["model_training_eligible"])

# Preserve the two event exclusions agreed in the original V2.7 methodology.
excluded_2026_keys = {"bahraininmalaysia", "unitedstates"}
excluded_mask = (work["year"] == 2026) & work["event_key"].isin(excluded_2026_keys)

eligible = work.loc[
    ~excluded_mask
    & ready
    & train_eligible
    & work["target"].isin([0, 1])
    & work["finishing_position_check"].notna()
    & work["starting_grid"].notna()
].copy()

eligible["event_id"] = (
    eligible["year"].astype(str) + "_" + eligible["event_key"].astype(str)
)

print("Source rows:", len(work))
print("Excluded rows under agreed rule:", int(excluded_mask.sum()))
print("Eligible rows with a usable starting grid:", len(eligible))
print("Eligible events:", eligible["event_id"].nunique())
display(eligible.groupby("year").agg(
    rows=("event_id", "size"),
    events=("event_id", "nunique"),
    winner_labels=("target", "sum")
).reset_index())

Source rows: 1354
Excluded rows under agreed rule: 0
Eligible rows with a usable starting grid: 1309
Eligible events: 64


,year,rows,events,winner_labels
0,2024,478,24,24
1,2025,479,24,24
2,2026,352,16,16


## 2. Validate one winner and unique driver rows per event

In [3]:
event_check = (
    eligible.groupby(["year", "event_id", "event_name"])
    .agg(
        driver_rows=("driver_code_norm", "size"),
        distinct_drivers=("driver_code_norm", "nunique"),
        winner_labels=("target", "sum"),
        distinct_grid_positions=("starting_grid", "nunique"),
        min_grid=("starting_grid", "min"),
        max_grid=("starting_grid", "max"),
    )
    .reset_index()
)
event_check["valid_event"] = (
    (event_check["driver_rows"] == event_check["distinct_drivers"])
    & (event_check["winner_labels"] == 1)
    & (event_check["driver_rows"] == event_check["distinct_grid_positions"])
)
display(event_check.loc[~event_check["valid_event"]])

if not event_check["valid_event"].all():
    raise ValueError("Some eligible events failed winner, driver uniqueness or grid-position checks.")

print("All eligible events passed event integrity checks.")

,year,event_id,event_name,driver_rows,distinct_drivers,winner_labels,distinct_grid_positions,min_grid,max_grid,valid_event


All eligible events passed event integrity checks.


## 4. Define the future-simulator feature schema

The selected features below are designed to be available when a user has supplied a hypothetical grid.

### Numeric features
- `starting_grid`
- `driver_prior_avg_finish_3`
- `driver_prior_avg_points_3`
- `driver_circuit_prior_avg_finish`
- `driver_circuit_prior_starts`
- `constructor_prior_avg_finish_3`
- `constructor_prior_avg_points_3`
- `constructor_circuit_prior_avg_finish`
- `constructor_circuit_prior_starts`

### Categorical features
- `driver_code_norm`
- `constructor`
- `event_norm`

### Deliberately excluded
- `round`: the future simulator does not ask the user to select a race round.
- Practice and qualifying lap/sector/time features: not supplied by the current simulator and may not correspond to a hypothetical grid.
- Practice/qualifying coverage flags: these describe target-event session availability, which is outside the current simulator input.
- `has_grid_data`: constant/near-constant after the user confirms a complete hypothetical grid.
- All race outcome and post-race fields.

In [4]:
NUMERIC_FEATURES = [
    "starting_grid",
    "driver_prior_avg_finish_3",
    "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish",
    "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
]

CATEGORICAL_FEATURES = [
    "driver_code_norm",
    "constructor",
    "event_norm",
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "target"
GROUP_COLUMN = "event_id"

missing_features = [c for c in FEATURE_COLUMNS if c not in eligible.columns]
if missing_features:
    raise ValueError(f"Required pre-race features are missing: {missing_features}")

print("Numeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Total model features:", len(FEATURE_COLUMNS))

Numeric features: 9
Categorical features: 3
Total model features: 12


## 5. Review feature missingness and historical coverage

In [13]:
feature_missingness = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "missing_count": [int(eligible[c].isna().sum()) for c in FEATURE_COLUMNS],
    "missing_percent": [
        round(float(eligible[c].isna().mean() * 100), 2)
        for c in FEATURE_COLUMNS
    ],
}).sort_values("missing_percent", ascending=False)

display(feature_missingness)

category_coverage = []
for col in CATEGORICAL_FEATURES:
    for season, group in eligible.groupby("year"):
        category_coverage.append({
            "feature": col,
            "season": int(season),
            "unique_values": int(group[col].nunique(dropna=True)),
            "values": ", ".join(sorted(group[col].dropna().astype(str).unique())),
        })
display(pd.DataFrame(category_coverage))

,feature,missing_count,missing_percent
3,driver_circuit_prior_avg_finish,648,49.50
7,constructor_circuit_prior_avg_finish,560,42.78
2,driver_prior_avg_points_3,28,2.14
1,driver_prior_avg_finish_3,28,2.14
6,constructor_prior_avg_points_3,24,1.83
5,constructor_prior_avg_finish_3,24,1.83
0,starting_grid,0,0.00
4,driver_circuit_prior_starts,0,0.00
8,constructor_circuit_prior_starts,0,0.00
9,driver_code_norm,0,0.00


,feature,season,unique_values,values
0,driver_code_norm,2024,24,"ALB, ALO, BEA, BOT, COL, DOO, GAS, HAM, HUL, L..."
1,driver_code_norm,2025,21,"ALB, ALO, ANT, BEA, BOR, COL, DOO, GAS, HAD, H..."
2,driver_code_norm,2026,23,"ALB, ALO, ANT, BEA, BOR, BOT, COL, GAS, HAD, H..."
3,constructor,2024,10,"Alpine F1 Team, Aston Martin, Ferrari, Haas F1..."
4,constructor,2025,10,"Alpine F1 Team, Aston Martin, Ferrari, Haas F1..."
5,constructor,2026,11,"Alpine F1 Team, Aston Martin, Audi, Cadillac F..."
6,event_norm,2024,24,"abudhabi, australian, austrian, azerbaijan, ba..."
7,event_norm,2025,24,"abudhabi, australian, austrian, azerbaijan, ba..."
8,event_norm,2026,16,"australian, austrian, azerbaijan, barcelona, b..."


## 3. Rebuild historical features using strictly prior race results

The source V2.6 dataset contains precomputed `*_prior_*` columns, but the original ETL formula is not included in this ZIP. To avoid relying on undocumented formulas, this notebook rebuilds the prior-performance features directly from the historical race results.

**Operational definitions used here:**
- Driver recent form: mean finishing position and race points over the driver's previous three eligible starts.
- Driver circuit form: mean finishing position and number of previous eligible starts at the same circuit.
- Constructor recent form: mean of the constructor's event-level average finishing position and points over its previous three eligible events.
- Constructor circuit form: mean constructor event-level finishing position and number of previous visits to the same circuit.

Every prior feature uses `shift(1)` or a prior-event cumulative count. The current event's result is therefore not included in its own features. The feature rows are then merged back to the event/driver records.

These definitions are the revised V2.8 methodology. They may differ from the undocumented V2.6 precomputed feature formulas.

In [12]:
# Use only eligible, completed historical events. Event-level integrity was
# checked in the previous cell, and the two agreed unmatched 2026 events
# have already been excluded from `eligible`.
history = eligible.copy()
history["event_date_sort"] = pd.to_datetime(history["event_date"], errors="coerce")
history["finish_num"] = pd.to_numeric(history["finishing_position"], errors="coerce")
history["points_num"] = pd.to_numeric(history["race_points"], errors="coerce")

if history["event_date_sort"].isna().any():
    raise ValueError("Some eligible rows have invalid event_date values.")
if history[["finish_num", "points_num"]].isna().any().any():
    raise ValueError("Historical finishing_position or race_points contains missing values.")

history = history.sort_values(
    ["event_date_sort", "year", "round", "event_id", "driver_code_norm"],
    kind="mergesort"
).copy()

# Driver's previous three starts (current event excluded).
driver_group = history.groupby("driver_code_norm", sort=False)
history["driver_prior_avg_finish_3"] = driver_group["finish_num"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)
history["driver_prior_avg_points_3"] = driver_group["points_num"].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)

# Driver's previous results at this same circuit (current event excluded).
driver_circuit_group = history.groupby(
    ["driver_code_norm", "event_norm"], sort=False
)
history["driver_circuit_prior_avg_finish"] = driver_circuit_group["finish_num"].transform(
    lambda s: s.shift(1).expanding(min_periods=1).mean()
)
history["driver_circuit_prior_starts"] = driver_circuit_group.cumcount()

# Build one constructor-level summary per event before calculating team form.
constructor_event = (
    history.groupby(
        ["event_id", "year", "round", "event_date_sort", "event_norm", "constructor"],
        dropna=False
    )
    .agg(
        constructor_event_avg_finish=("finish_num", "mean"),
        constructor_event_avg_points=("points_num", "mean"),
    )
    .reset_index()
    .sort_values(
        ["event_date_sort", "year", "round", "event_id", "constructor"],
        kind="mergesort"
    )
)

# Constructor's previous three event appearances (current event excluded).
constructor_group = constructor_event.groupby("constructor", sort=False)
constructor_event["constructor_prior_avg_finish_3"] = constructor_group[
    "constructor_event_avg_finish"
].transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
constructor_event["constructor_prior_avg_points_3"] = constructor_group[
    "constructor_event_avg_points"
].transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())

# Constructor's previous visits to this circuit (current event excluded).
constructor_circuit_group = constructor_event.groupby(
    ["constructor", "event_norm"], sort=False
)
constructor_event["constructor_circuit_prior_avg_finish"] = constructor_circuit_group[
    "constructor_event_avg_finish"
].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
constructor_event["constructor_circuit_prior_starts"] = constructor_circuit_group.cumcount()

constructor_features = constructor_event[[
    "event_id", "constructor",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
]].copy()

history = history.drop(columns=[
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
], errors="ignore")

eligible = history.merge(
    constructor_features,
    on=["event_id", "constructor"],
    how="left",
    validate="many_to_one"
)

# Retain deterministic chronological order after the merge.
eligible = eligible.sort_values(
    ["year", "round", "event_id", "starting_grid"],
    kind="mergesort"
).reset_index(drop=True)

# The first historical appearances correctly remain missing.
assert eligible.loc[
    eligible.groupby("driver_code_norm").cumcount().eq(0),
    "driver_prior_avg_finish_3"
].isna().all()

print("Rebuilt prior features:", [
    "driver_prior_avg_finish_3",
    "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish",
    "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
])
print("Current-event results excluded using shifted historical sequences.")

Rebuilt prior features: ['driver_prior_avg_finish_3', 'driver_prior_avg_points_3', 'driver_circuit_prior_avg_finish', 'driver_circuit_prior_starts', 'constructor_prior_avg_finish_3', 'constructor_prior_avg_points_3', 'constructor_circuit_prior_avg_finish', 'constructor_circuit_prior_starts']
Current-event results excluded using shifted historical sequences.


In [7]:
prior_features = [
    "driver_prior_avg_finish_3",
    "driver_prior_avg_points_3",
    "driver_circuit_prior_avg_finish",
    "driver_circuit_prior_starts",
    "constructor_prior_avg_finish_3",
    "constructor_prior_avg_points_3",
    "constructor_circuit_prior_avg_finish",
    "constructor_circuit_prior_starts",
]

if "event_date" in eligible.columns:
    eligible["event_date_check"] = pd.to_datetime(eligible["event_date"], errors="coerce")
    print("Event dates parsed:", int(eligible["event_date_check"].notna().sum()), "/", len(eligible))
else:
    print("WARNING: event_date is unavailable; temporal provenance cannot be checked from this dataset.")

prior_review = pd.DataFrame({
    "feature": prior_features,
    "available_non_null": [int(eligible[c].notna().sum()) for c in prior_features],
    "missing_percent": [round(float(eligible[c].isna().mean() * 100), 2) for c in prior_features],
    "requires_ETL_formula_verification": True,
})
display(prior_review)

Event dates parsed: 1309 / 1309


,feature,available_non_null,missing_percent,requires_ETL_formula_verification
0,driver_prior_avg_finish_3,1281,2.14,True
1,driver_prior_avg_points_3,1281,2.14,True
2,driver_circuit_prior_avg_finish,661,49.50,True
3,driver_circuit_prior_starts,1309,0.00,True
4,constructor_prior_avg_finish_3,1285,1.83,True
5,constructor_prior_avg_points_3,1285,1.83,True
6,constructor_circuit_prior_avg_finish,749,42.78,True
7,constructor_circuit_prior_starts,1309,0.00,True


## 6. Leakage audit

In [14]:
FORBIDDEN_INPUTS = {
    "target", "is_winner", "finishing_position", "finishing_position_check",
    "race_points", "race_status", "winner_label_count",
    "event_prediction_ready", "model_training_eligible",
    "event_driver_count", "qualifying_feature_count", "grid_feature_count",
    "qualifying_coverage", "grid_coverage", "feature_match_status",
    "event_prediction_ready_check", "model_training_eligible_check",
    "event_id", "event_name", "event_date", "year",
}

leakage_hits = sorted(set(FEATURE_COLUMNS) & FORBIDDEN_INPUTS)
print("Forbidden fields in model features:", leakage_hits)
assert not leakage_hits, "A forbidden outcome/eligibility field is included in the model input."

assert "starting_grid" in FEATURE_COLUMNS
assert not any("practice" in c or "qualifying" in c or "quali_" in c for c in FEATURE_COLUMNS)
assert "round" not in FEATURE_COLUMNS

print("Structural feature-schema audit passed.")
print("Reminder: prior-feature temporal provenance still requires source-ETL verification.")

Forbidden fields in model features: []
Structural feature-schema audit passed.
Reminder: prior-feature temporal provenance still requires source-ETL verification.


## 7. Preserve chronological train / validation / final-test split

In [15]:
train_df = eligible.loc[eligible["year"] == 2024].copy()
valid_df = eligible.loc[eligible["year"] == 2025].copy()
test_df = eligible.loc[eligible["year"] == 2026].copy()

split_summary = pd.DataFrame([
    {"split": "Train", "season": 2024, "rows": len(train_df), "events": train_df[GROUP_COLUMN].nunique()},
    {"split": "Validation", "season": 2025, "rows": len(valid_df), "events": valid_df[GROUP_COLUMN].nunique()},
    {"split": "Final test", "season": 2026, "rows": len(test_df), "events": test_df[GROUP_COLUMN].nunique()},
])
display(split_summary)

if train_df.empty or valid_df.empty or test_df.empty:
    raise ValueError("One or more chronological splits are empty.")

print("2026 remains reserved for final testing; do not use it for model selection.")

,split,season,rows,events
0,Train,2024,478,24
1,Validation,2025,479,24
2,Final test,2026,352,16


2026 remains reserved for final testing; do not use it for model selection.


## 8. Compare the starting-grid P1 baseline

In [16]:
def starting_grid_baseline(frame, split_name):
    results = []
    for event_id, group in frame.groupby(GROUP_COLUMN):
        group = group.dropna(subset=["starting_grid"]).copy()
        winners = group.loc[group[TARGET_COLUMN] == 1]
        if group.empty or len(winners) != 1:
            continue
        pole = group.loc[group["starting_grid"].idxmin()]
        winner = winners.iloc[0]
        results.append({
            "split": split_name,
            "event_id": event_id,
            "actual_winner": winner["driver_code"],
            "predicted_winner": pole["driver_code"],
            "winner_grid": float(winner["starting_grid"]),
            "top1_correct": bool(winner["driver_code_norm"] == pole["driver_code_norm"]),
            "top3_correct": bool(winner["starting_grid"] <= 3),
        })
    details = pd.DataFrame(results)
    summary = {
        "split": split_name,
        "events": len(details),
        "top1_accuracy_pct": details["top1_correct"].mean() * 100 if len(details) else np.nan,
        "top3_accuracy_pct": details["top3_correct"].mean() * 100 if len(details) else np.nan,
    }
    return summary, details

baseline_summaries = []
baseline_details = {}
for split_name, frame in [("2025 validation", valid_df), ("2026 final test", test_df)]:
    summary, details = starting_grid_baseline(frame, split_name)
    baseline_summaries.append(summary)
    baseline_details[split_name] = details

display(pd.DataFrame(baseline_summaries))

,split,events,top1_accuracy_pct,top3_accuracy_pct
0,2025 validation,24,66.666667,95.833333
1,2026 final test,16,68.750000,93.750000


## 9. Create the V2.8 pre-race modelling dataset and manifest

In [17]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

keep_columns = list(dict.fromkeys([
    "dataset_version", "year", "round", "event_date", "event_name", "event_norm",
    "driver_code", "driver_code_norm", "constructor",
    "starting_grid", "finishing_position", "is_winner", "race_points", "race_status",
    "event_prediction_ready", "model_training_eligible",
    "target", "event_id",
] + FEATURE_COLUMNS))

dataset_out = eligible[keep_columns].copy()
dataset_out.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow")

manifest = {
    "project": "For1Sight",
    "version": "V2.8",
    "stage": "Stage 2 - pre-race feature engineering",
    "source_dataset": str(SOURCE_PATH),
    "output_dataset": str(OUTPUT_PATH),
    "historical_seasons": [2024, 2025, 2026],
    "training_season": 2024,
    "validation_season": 2025,
    "final_test_season": 2026,
    "excluded_2026_event_keys": sorted(excluded_2026_keys),
    "prediction_objective": "race_winner_classification",
    "prediction_input": "user_supplied_hypothetical_starting_grid",
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "feature_columns": FEATURE_COLUMNS,
    "prior_feature_temporal_provenance_verified": True,
    "notes": [
        "Practice and qualifying features excluded to match current simulator inputs.",
        "Round excluded because the future prediction UI does not require race-round input.",
        "Prior driver and constructor features were independently rebuilt from earlier eligible event results using shifted chronological sequences.",
        "2026 remains reserved for final testing and must not be used for model selection."
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved dataset:", OUTPUT_PATH)
print("Saved manifest:", MANIFEST_PATH)
print("Rows:", len(dataset_out))
print("Events:", dataset_out["event_id"].nunique())
print("Features:", len(FEATURE_COLUMNS))

Saved dataset: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_8_prerace.parquet
Saved manifest: C:\Users\ayepd\Desktop\For1Sight Project\for1sight_warehouse\prediction_training_v2_8_prerace_manifest.json
Rows: 1309
Events: 64
Features: 12


## 10. Stage 2 sign-off checklist

- [ ] Source dataset loaded successfully.
- [ ] Agreed event eligibility rules reproduced.
- [ ] Every event has exactly one winner.
- [ ] No duplicate driver rows or duplicate grid positions within an event.
- [ ] Model inputs match the future simulator's available information.
- [ ] Practice and qualifying fields are excluded.
- [ ] Race round is excluded from model features.
- [ ] Outcome and post-race fields are excluded from model inputs.
- [ ] Prior driver and constructor features rebuilt with current-event results excluded.
- [ ] Missingness and category coverage reviewed.
- [ ] Starting-grid P1 baseline calculated.
- [ ] 2024 / 2025 / 2026 chronological split preserved.
- [ ] V2.8 dataset and manifest saved successfully.

**Next:** Stage 3 training using exactly the feature schema in the saved manifest. Keep 2026 untouched until final evaluation.